# YOLOv8n · detectar paneles y clasificar imágenes

Abrir en **Colab con GPU T4**, en una sesión nueva, y ejecutar de arriba hacia abajo.
Se usan las 900 imágenes existentes y el split oficial 738/162. No se agregan imágenes.

Se comparan dos corridas **independientes**, ambas desde pesos COCO `yolov8n.pt`:
entrenar toda la red y congelar los primeros diez módulos (backbone). Se mantiene la
misma receta entre variantes; no es una comparación controlada con las CNN de clasificación.

Las etiquetas se preparan en una **copia local RGB**: clase única `panel` (0), excluir
solo Z01/línea 1, conservar cajas pequeñas y confianza original como metadatos.
N06/N16 conservan su referencia provisional. Los originales no se modifican.

**Val elige el checkpoint y también se usa para reportar**: no es un test independiente.
Se calculan mAP50/mAP50-95 y curvas PR sobre todo val y sobre Jamaica (23 imágenes).
Como clasificador se usa la máxima confianza de las cajas; no es probabilidad calibrada.
El umbral inicial 0.25 es exploratorio y no se optimiza aquí.

Drive: `Mi unidad/PDI-TP/resultados/yolov8n/<modo>/`, con `modelo.pt`, métricas,
predicciones, cajas y subcarpetas de entrenamiento/evaluación. Dejar espacio para ambas
corridas. Los checkpoints `best.pt` y `last.pt` se escriben en Drive en cada época;
si falla la sesión se recupera hasta la última época guardada.

In [ ]:
# 1. Opciones: comenzar con ambas variantes para cubrir la comparación de la consigna.
REPETIR = False  # True archiva la corrida anterior y vuelve a entrenar.
MODOS = {"coco_completo": 0, "coco_backbone_congelado": 10}
SEMILLA = 42
LADO = 256
BATCH = 16
EPOCAS = 50
PACIENCIA = 10
UMBRAL = 0.25
CONF_MIN = 0.001  # Bajo para calcular PR/mAP; distinto del umbral de clasificación.
IOU_NMS = 0.7
MAX_DETECCIONES = 300
assert 0 < CONF_MIN < UMBRAL <= 1

In [ ]:
# 2. Código y dependencias: entorno PyTorch separado de los notebooks TensorFlow.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Abrir este notebook en Google Colab con GPU T4.")

repo = Path("/content/PDI-TP")
if repo.is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(repo / "requirements-detection.txt")], check=True)

import gc
import json
import platform
import shutil
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from pdi_tp import entorno, resultados, deteccion, clasificacion as clf

if version("ultralytics") != "8.3.221":
    raise RuntimeError("Reiniciar la sesión tras instalar dependencias y volver a ejecutar.")
if not torch.cuda.is_available():
    raise RuntimeError("Activar GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.")
rutas = entorno.preparar()
print("GPU:", torch.cuda.get_device_name(0))

## Preparar las etiquetas sin cambiar el dataset

La auditoría frena ante errores nuevos. Los negativos tienen un `.txt` vacío y también
entran en el entrenamiento. Se guardan manifiesto y cajas con clase/confianza originales.
La copia derivada se valida antes de reutilizarla; queda fuera de Drive.

In [ ]:
DERIVADO = repo / "data" / "yolo_paneles_v1"
tabla, cajas = deteccion.preparar_dataset(rutas, DERIVADO)
HUELLA = clf.huella_tabla(tabla)
val = tabla.query("split == 'val'").reset_index(drop=True)
archivos_val = val.archivo.to_list()
fuentes_val = [str(DERIVADO / "images" / "val" / n) for n in archivos_val]
display(tabla.groupby(["split", "fuente", "y_original"]).size().rename("imagenes").to_frame())
print("Cajas incluidas:", int(cajas.incluida.sum()), "Excluidas:", int((~cajas.incluida).sum()))
print("Datos:", DERIVADO / "datos.yaml")

RECETA = dict(imgsz=LADO, batch=BATCH, epochs=EPOCAS, patience=PACIENCIA,
    optimizer="AdamW", lr0=0.001, lrf=0.01, momentum=0.9, weight_decay=0.0005,
    warmup_epochs=3.0, seed=SEMILLA, deterministic=True, workers=2,
    amp=True, cache=False, rect=False, multi_scale=False, fraction=1.0,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, degrees=90.0, translate=0.1, scale=0.2,
    shear=0.0, perspective=0.0, flipud=0.5, fliplr=0.5,
    mosaic=0.0, mixup=0.0, copy_paste=0.0, close_mosaic=0,
    box=7.5, cls=0.5, dfl=1.5,
    conf=CONF_MIN, iou=IOU_NMS, max_det=MAX_DETECCIONES,
    save=True, save_period=-1, plots=True, val=True, device=0)
ENTORNO = dict(python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
    paquetes={p: version(p) for p in ("ultralytics", "torch", "torchvision", "numpy", "pandas")})

## Entrenar, evaluar y guardar

Cada variante completa se saltea. Una corrida incompleta retoma `last.pt` si conserva
optimizador; si el entrenamiento terminó, reutiliza `best.pt` y completa la evaluación.
Cambiar la receta exige `REPETIR = True`; se conservan las corridas anteriores.

`freeze=10` congela los módulos 0–9 del backbone de YOLOv8n; `freeze=0` permite ajustar
todos los módulos entrenables (Ultralytics mantiene fija la proyección DFL).
El mejor checkpoint se selecciona por el criterio nativo de detección de esta versión
de Ultralytics. La receta efectiva completa queda en `entrenamiento/args.yaml`.

La evaluación usa confianza mínima 0.001 para conservar cajas de baja confianza al
construir PR/mAP. La clasificación aplica después el umbral 0.25 al máximo por imagen.
Se guardan todas las cajas tras NMS en `cajas_val.csv`, incluso las inferiores al umbral.

In [ ]:
def evaluar_detector(modelo, carpeta, yaml_datos, nombre):
    evaluacion = modelo.val(data=str(yaml_datos), split="val", imgsz=LADO, batch=BATCH,
        device=0, workers=2, conf=CONF_MIN, iou=IOU_NMS, max_det=MAX_DETECCIONES,
        augment=False, half=False, plots=True, project=str(carpeta), name=nombre, exist_ok=True)
    resumen = dict(map50=float(evaluacion.box.map50), map50_95=float(evaluacion.box.map),
                   precision=float(evaluacion.box.mp), recall=float(evaluacion.box.mr))
    if not np.isfinite(list(resumen.values())).all():
        raise RuntimeError("La evaluación de detección contiene métricas no finitas.")
    return resumen


for modo, congelar in MODOS.items():
    carpeta = resultados.ruta_corrida(rutas, "yolov8n", modo)
    if resultados.corrida_completa(carpeta) and not REPETIR:
        print("Corrida completa, se reutiliza:", carpeta)
        continue
    carpeta = resultados.carpeta_corrida(rutas, "yolov8n", modo, reiniciar=REPETIR)
    params = dict(arquitectura="yolov8n", inicializacion="COCO yolov8n.pt", freeze=congelar,
        receta=RECETA, huella_manifiesto=HUELLA, entorno=ENTORNO,
        criterio_datos="v1: clase panel; excluir solo Z01/linea 1; mantener cajas pequenas",
        evaluacion="val selecciona checkpoint y reporta; N06/N16 provisionales",
        puntaje_clasificacion="maxima confianza de cajas tras NMS; 0 sin detecciones; no calibrado",
        umbral_clasificacion=UMBRAL)
    # Evita mezclar entrenamiento reanudado con otra receta o dataset.
    solicitud = carpeta / "solicitud.json"
    if solicitud.exists() and json.loads(solicitud.read_text()) != params:
        raise RuntimeError("Cambió la configuración. Usar REPETIR=True para archivar la anterior.")
    solicitud.write_text(json.dumps(params, indent=2), encoding="utf-8")
    shutil.copy2(DERIVADO / "manifiesto.csv", carpeta / "manifiesto.csv")
    shutil.copy2(DERIVADO / "cajas_derivadas.csv", carpeta / "cajas_derivadas.csv")
    entrenamiento = carpeta / "entrenamiento"
    ultimo = entrenamiento / "weights" / "last.pt"
    mejor = entrenamiento / "weights" / "best.pt"
    if ultimo.exists():
        # Solo checkpoints propios creados por este notebook.
        estado = torch.load(ultimo, map_location="cpu", weights_only=False)
        epoca = int(estado["epoch"])
        del estado
        if epoca >= 0:
            if not mejor.exists():
                raise RuntimeError("Falta best.pt: conservar archivos y revisar antes de reanudar.")
            print("Reanudando desde last.pt:", modo)
            modelo = YOLO(str(ultimo))
            modelo.train(resume=True, data=str(DERIVADO / "datos.yaml"), device=0)
        else:
            print("Entrenamiento finalizado; se recupera best.pt para evaluar:", modo)
    elif mejor.exists() or (entrenamiento / "results.csv").exists():
        raise RuntimeError("Hay historial o best.pt pero falta last.pt; no se sobrescribe la corrida.")
    else:
        modelo = YOLO("yolov8n.pt")
        modelo.train(data=str(DERIVADO / "datos.yaml"), project=str(carpeta),
                     name="entrenamiento", exist_ok=True, freeze=congelar, **RECETA)
    if not mejor.is_file():
        raise RuntimeError("No se generó best.pt; revisar el entrenamiento.")
    shutil.copy2(mejor, carpeta / "modelo.pt")
    modelo = YOLO(str(carpeta / "modelo.pt"))
    metricas_det = {
        "val": evaluar_detector(modelo, carpeta, DERIVADO / "datos.yaml", "deteccion_val"),
        "jamaica": evaluar_detector(modelo, carpeta, DERIVADO / "datos_jamaica.yaml", "deteccion_jamaica"),
    }
    salidas = modelo.predict(source=fuentes_val, stream=True, imgsz=LADO, batch=BATCH,
        device=0, conf=CONF_MIN, iou=IOU_NMS, max_det=MAX_DETECCIONES,
        augment=False, half=False, save=False, verbose=False)
    puntajes, filas = deteccion.resumir_predicciones(salidas, archivos_val)
    pd.DataFrame(filas, columns=["archivo", "clase", "confianza", "xc", "yc", "w", "h"]).to_csv(
        carpeta / "cajas_val.csv", index=False)
    clf.guardar_json(carpeta / "metricas_deteccion.json", metricas_det)
    mascara = ~val.pendiente_revision.to_numpy()
    extra = dict(deteccion=metricas_det,
        sin_pendientes=clf.evaluar(val.loc[mascara, "y_original"].to_numpy(dtype=int),
                                   puntajes[mascara], UMBRAL))
    resultados.guardar_corrida(rutas, carpeta, archivos=archivos_val, probabilidad=puntajes,
                              params=params, umbral=UMBRAL, metricas_extra=extra)
    display(pd.DataFrame(metricas_det).T)
    del modelo
    gc.collect()
    torch.cuda.empty_cache()

## Revisar resultados reales y ejemplos

Las curvas PR, matrices y ejemplos de detecciones quedan en `deteccion_val/` y
`deteccion_jamaica/`; el historial y las curvas de entrenamiento, en `entrenamiento/`.
Las matrices de Ultralytics son de objetos y fondo, no de imágenes con/sin paneles.
La matriz binaria siguiente usa las predicciones del formato común.

Con solo 11 positivos y 12 negativos de Jamaica en val, una imagen cambia mucho el
recall: no elegir un ganador por diferencias pequeñas. El umbral definitivo y la
validación cruzada de Jamaica siguen pendientes del acuerdo del equipo.

In [ ]:
from IPython.display import Image as Imagen, display
from sklearn.metrics import ConfusionMatrixDisplay

corridas = resultados.cargar_corridas(rutas)
columnas = ["modelo", "modo", "f1_macro", "recall", "recall_sin_paneles",
            "jamaica.f1_macro", "jamaica.recall", "jamaica.recall_sin_paneles"]
display(corridas.loc[corridas.modelo.isin(["yolov8n", "resnet50"])].reindex(columns=columnas))
for modo in MODOS:
    carpeta = resultados.ruta_corrida(rutas, "yolov8n", modo)
    metricas = json.loads((carpeta / "metricas.json").read_text())
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(np.array(metricas["matriz_confusion"]),
        display_labels=["sin paneles", "con paneles"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"YOLOv8n/{modo} · clasificación provisional")
    fig.tight_layout()
    fig.savefig(carpeta / "matriz_clasificacion.png", dpi=130)
    plt.show()
    plt.close(fig)
    print(modo, "· detección:")
    display(pd.DataFrame(json.loads((carpeta / "metricas_deteccion.json").read_text())).T)
    ejemplo = carpeta / "deteccion_val" / "val_batch0_pred.jpg"
    if ejemplo.is_file():
        display(Imagen(filename=str(ejemplo)))
    print("Guardado en:", carpeta)

Referencias: [YOLOv8](https://docs.ultralytics.com/models/yolov8/),
[entrenamiento y reanudación](https://docs.ultralytics.com/modes/train/),
[validación](https://docs.ultralytics.com/modes/val/).
Implementación fijada en `ultralytics==8.3.221`; las versiones efectivas de PyTorch
y del resto del entorno se registran en cada corrida. **No hay resultados simulados.**